# TRPG 룰북 RAG

## ⚙️ 설정값

In [ ]:
# 설정값 (바꿀 값은 여기만)

OLLAMA_URL = "http://localhost:11434"     # Ollama 서버 주소
EMBED_MODEL = "bge-m3"                    # 임베딩 모델 (글 → 숫자 벡터)
GEN_MODEL = "qwen2.5:14b"                 # 번역·선택·검증 모델
TRANSLATE_MODEL = GEN_MODEL               # 번역도 같은 14B 사용

REPO = "https://raw.githubusercontent.com/TRPG-masterbot/data/main/Min"
LOCAL = "/content"                               # 깃허브에서 받은 파일을 둘 곳 (colab에)
DATA = "/content/drive/MyDrive/KT 기본프로젝트"

OUT_PATH = f"{LOCAL}/rules_new.jsonl"         # 룰북 청크 파일
DEV_PATH = f"{LOCAL}/golden_dev2.json"        # 개발 질문 세트 149문항
FINAL_PATH = f"{DATA}/golden_final.json"      # 최종 질문 세트
EMB_PATH = f"{DATA}/rules_new_emb.npy"        # 임베딩 저장 파일
CACHE_PATH = f"{DATA}/cache_rules.json"       # 번역·검색 캐시

# ── 검색 ──
TOP_K = 5                                 # 최종으로 가져올 청크 수
POOL_K = 10                               # 검색 하나당 후보 수 (합치기 전)
SEARCH_WEIGHTS = [1, 1, 1, 0.5, 0.5]      # 검색 비중: 임베딩 3개, 키워드 2개
RRF_C = 60                                # RRF 점수 = 비중 / (RRF_C + 순위)
NUM_CTX = 8192                            # 모든 LLM 호출 공통 num_ctx (같아야 모델을 다시 안 올림)

# ── 근거 고르기 ──
GEN_SEED = 42                             # 같은 질문 → 같은 결과
SELECT_CHARS = 300                        # 근거 고를 때 후보마다 보여 줄 글자 수
VERIFY_CHARS = 800                        # 근거 검증 때 보여 줄 글자 수

# ── 동시 처리 ──
NUM_PARALLEL = 4                          # 모델 하나가 동시에 처리할 요청 수 (L4 기준 4: 14B 9GB + 대화 칸 1.6GB × 4)

## 0. 설치

In [ ]:
%%writefile /content/requirements_rb.txt
pymupdf==2.0
requests==2.32.4
chromadb==1.5.9
langgraph==1.2.12

In [ ]:
# 설치
!pip install -r /content/requirements_rb.txt -q
!pip freeze | grep -E "pymupdf|requests|chromadb|langgraph"

## 1. Ollama 설치 · 서버 · 모델 받기
모델은 **백그라운드로** 받아요. 다운로드가 도는 동안 다음 셀들을 진행해도 돼요.

In [ ]:
!apt-get install -y -qq zstd
!curl -fsSL http://ollama.com/install.sh | sh
!ollama --version

In [ ]:
# import · 서버 켜기
import os, re, json, time, math, subprocess
from collections import Counter
import numpy as np, requests, chromadb
from tqdm.auto import tqdm
from google.colab import drive, files
from concurrent.futures import ThreadPoolExecutor
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

def wait(url, seconds=120):         # 서버가 켜질 때까지 1초마다 확인
    for _ in range(seconds):
        try:
            if requests.get(url, timeout=2).status_code == 200:
                print("서버 준비 완료:", url)
                return
        except Exception:
            pass                          # 아직 안 켜졌으면 에러 → 무시하고 다시
        time.sleep(1)
    print("서버가 아직 준비되지 않았습니다. ollama.log를 확인하세요")


os.environ["OLLAMA_KEEP_ALIVE"] = "1h"    # 모델을 1시간 동안 메모리에 유지
os.environ["OLLAMA_NUM_PARALLEL"] = str(NUM_PARALLEL)   # 요청을 동시에 처리
subprocess.Popen("nohup ollama serve > ollama.log 2>&1 &", shell=True)   # 서버를 백그라운드로 켜기
wait(OLLAMA_URL)

In [ ]:
# 모델 받기
subprocess.Popen(f'nohup sh -c "ollama pull {EMBED_MODEL} && ollama pull {GEN_MODEL}" > pull.log 2>&1 &', shell=True)
print("다운로드 시작. 진행 확인은 아래 셀")

In [ ]:
# 모델 대기
t0 = time.time()
ready = False

while time.time() - t0 < 1200:
    out = subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout   # 받은 모델 목록
    if GEN_MODEL in out and EMBED_MODEL in out:
        print(f"모델 준비 완료 ({int(time.time() - t0)}초 기다림)")
        ready = True
        break
    print("아직 받는 중...", int(time.time() - t0), "초")
    time.sleep(15)                                   # 15초마다 확인

if not ready:
    print("20분이 지나도 안 받아졌어요. pull.log를 확인하세요:")
    print(open("pull.log").read()[-500:])            # 로그 끝부분 500글자

## 2. 구글 드라이브 · 경로

In [ ]:
# 드라이브 연결
drive.mount("/content/drive")
print(OLLAMA_URL, "| LLM(번역·생성 공통):", GEN_MODEL, "| 임베딩:", EMBED_MODEL)

In [ ]:
# 데이터 받기 (깃허브 → /content)
for name in ["rules_new.jsonl", "golden_dev2.json"]:
    res = requests.get(f"{REPO}/{name}", timeout=60)
    res.raise_for_status()                         # 파일 이름이나 주소가 틀리면 여기서 멈춤
    with open(f"{LOCAL}/{name}", "wb") as f:
        f.write(res.content)
    print("받음:", name, len(res.content), "bytes")

## 1. 청크 읽기 · 임베딩 · Chroma 컬렉션

In [ ]:
# Ex01-a. 청크 읽기
chunks = []
for line in open(OUT_PATH, encoding="utf-8"):   # jsonl = 한 줄에 청크 하나
    c = json.loads(line)
    if c["meta"]["type"] == "rule":              # 규칙 청크만 검색 대상
        chunks.append(c)

print("검색 대상 청크:", len(chunks))

In [ ]:
# Ex01-b. 임베딩

def embed(texts, model=EMBED_MODEL):   # 글 여러 개 → 벡터 여러 개
    res = requests.post(
        f"{OLLAMA_URL}/api/embed",
        json={"model": model, "input": texts, "options": {"num_ctx": 8192}},
        timeout=300,
    )
    res.raise_for_status()                       # 서버 오류면 여기서 멈춤
    return res.json()["embeddings"]


BATCH = 32

# 저장된 임베딩이 있고 청크 수도 같으면 → 다시 계산하지 않고 불러오기
saved = np.load(EMB_PATH) if os.path.exists(EMB_PATH) else None

if saved is not None and saved.shape[0] == len(chunks):
    emb = saved
    print("캐시 사용:", emb.shape)
else:
    vecs = []
    for i in tqdm(range(0, len(chunks), BATCH)):
        batch = [c["text"] for c in chunks[i:i + BATCH]]
        vecs += embed(batch)
    emb = np.array(vecs, dtype="float32")
    np.save(EMB_PATH, emb)
    print("새로 계산:", emb.shape)

# 캐시 사용: (2180, 1024)

In [ ]:
# Ex01-c. Chroma 컬렉션
# 임베딩은 Ex01-b에서 만든 것을 그대로 넣음
client = chromadb.Client()                     # 메모리에만 만듦

# 같은 런타임에서 다시 실행해도 옛 벡터가 남지 않게, 있으면 지우고 새로 만듦
try:
    client.delete_collection("rules")
except Exception:
    pass                                       # 처음 실행이면 지울 게 없어서 에러 → 무시

collection = client.create_collection(
    name="rules",
    configuration={"hnsw": {"space": "cosine",    # 코사인 유사도로 비교
                            "ef_search": 200}},   # 검색할 때 더 넓게 살펴봄
)

ids = [str(i) for i in range(len(chunks))]     # Chroma id는 문자열: 청크 순서 번호를 그대로 사용
metadatas = [{"h1": c["meta"]["h1"]} for c in chunks]     # 장 이름
collection.add(ids=ids, embeddings=emb.tolist(), metadatas=metadatas)
print("컬렉션에 들어간 벡터 수:", collection.count())

# 컬렉션에 들어간 벡터 수: 2180

## 2. 평가 도구 · 질문 세트
질문 세트는 `{"q": 질문, "want": 정답 제목}`. 검색에는 `q`만, `want`는 채점에만 써요. `want`가 `null`이면 룰북에 답이 없는 거절 문항이에요.

In [ ]:
# Ex02. 제목 비교

def norm(title):                    # 제목 다듬기 (꼬리표 제거 + 소문자)
    title = re.sub(r"\s*\[[^\]]*\]$", "", title)    # 예: "Prone [Condition]" → "Prone"
    title = re.sub(r"\s*\([^)]*\)$", "", title)     # 예: "Hide (Action)"     → "Hide"
    return title.strip().lower()


def hit(title, want):               # 정답인지 (정확한 비교)
    return norm(title) == norm(want)


def stem(title):                    # 단수/복수 차이 없애기 (단어 끝 s 제거)
    return re.sub(r"s\b", "", norm(title))          # 예: "attacks" → "attack"


def hit2(title, want):              # 정답인지 (느슨한 비교: 단수/복수 차이 무시)
    return stem(title) == stem(want)


# 확인
print(norm("Prone [Condition]"), "|", hit("Prone [Condition]", "Prone"))
print(hit("Attacks", "Attack"), "|", hit2("Attacks", "Attack"))

# prone | True
# False | True

In [ ]:
# Ex03. 채점

def wants(item):                    # 정답을 항상 리스트로 ("Prone" → ["Prone"], 정답 없음 → [])
    want = item["want"]
    if want is None:                               # 거절 문항
        return []
    if isinstance(want, str):
        return [want]
    return want


def rank(found, item):              # 정답 등수 찾기 (없으면 None)
    titles = [c["meta"]["title"] for c in found]
    for i, title in enumerate(titles):
        if any(hit2(title, w) for w in wants(item)):
            return i + 1, titles                    # 등수(1부터), 검색된 제목 목록
    return None, titles


def counts(ranks):                  # 순위 세기 → (1등, 3등 안, 5등 안)
    top1 = sum(1 for r in ranks if r == 1)
    top3 = sum(1 for r in ranks if r is not None and r <= 3)
    top5 = sum(1 for r in ranks if r is not None)   # 검색이 5개만 가져오므로 "찾기만 하면" 5등 안
    return (top1, top3, top5)


def evaluate(search, items, misses=False):   # 질문 세트 전체 채점 (search = 검색 함수)
    ranks = []
    for g in tqdm(items, leave=False):
        found = search(g["q"])                      # 검색에는 질문(q)만
        r, titles = rank(found, g)                  # 정답(want)은 채점에만
        ranks.append(r)
        if misses and r != 1:                       # 1등 못 한 질문 출력
            print(f"[{r}] {g['q']}\n     정답: {g['want']} | 검색: {titles[:3]}")
    return counts(ranks)

# 확인 (가짜 검색 결과로)
fake = [{"meta": {"title": t}} for t in ["Shove", "Prone [Condition]", "Grappled"]]
print(rank(fake, {"want": "Prone"})[0], "|", counts([1, 2, None, 5]))

# 2 | (1, 2, 3)

In [ ]:
# Ex04. 질문 세트
def load(path):                     # 질문 세트 파일 읽기 (없으면 빈 리스트)
    if not os.path.exists(path):
        return []
    return json.load(open(path, encoding="utf-8"))

devset   = load(DEV_PATH)           # 개발 149문항 (정답 없는 거절 문항 포함)
finalset = load(FINAL_PATH)         # 최종 60문항

# 정답 제목에 오타가 있으면 영원히 0점 → 룰북 제목 목록과 대조
known = {norm(c["meta"]["title"]) for c in chunks}

for name, items in {"devset": devset, "finalset": finalset}.items():
    miss = []
    for g in items:
        for w in wants(g):
            if norm(w) not in known :
                miss.append(w)
    print(f"{name}: {len(items)}문항 | 룰북에 없는 정답: {miss}")

## 3. 용어표
룰북에서 이름이 정해진 규칙 용어만 (일반 단어 제외). 번역②의 용어 목록을 여기서 뽑고, GM 답변의 한국어 표기 통일에도 써요.

In [ ]:
# Ex05. 용어표
# 한국어 = 룰북 영어 용어. 번역②의 용어 목록과 GM 답변의 표기 통일에 같이 씀
# 기준: 룰북에서 이름이 정해진 "규칙 용어"만 넣음 (번역이 제각각이 되기 쉬운 것)
#       주문·속도·피해·무기처럼 누가 번역해도 같은 일반 단어는 넣지 않음
TERM_V2 = [
    # ── 판정·전투 ──
    ("이점", "Advantage"), ("불리점", "Disadvantage"), ("능력 판정", "Ability Check"),
    ("공격 굴림", "Attack Roll"), ("내성 굴림", "Saving Throw"), ("기회 공격", "Opportunity Attack"),
    ("반응", "Reaction"), ("추가 행동", "Bonus Action"), ("집중", "Concentration"),
    ("이니셔티브", "Initiative"), ("저항", "Resistance"), ("취약", "Vulnerability"), ("면역", "Immunity"),
    ("엄폐", "Cover"), ("치명타", "Critical Hit"), ("숙련", "Proficiency"), ("전문화", "Expertise"),
    ("긴 휴식", "Long Rest"), ("짧은 휴식", "Short Rest"), ("D20 판정", "D20 Test"),
    ("난이도 등급", "Difficulty Class"), ("방어도", "Armor Class"), ("피해 굴림", "Damage Roll"),
    ("죽음 내성 굴림", "Death Saving Throw"), ("체력", "Hit Points"), ("체력 주사위", "Hit Point Dice"),
    ("임시 체력", "Temporary Hit Points"), ("피투성이", "Bloodied"), ("안정", "Stable"),
    ("기습", "Surprise"), ("붙잡기", "Grappling"), ("맨손 타격", "Unarmed Strike"),
    ("주문 공격", "Spell Attack"), ("무기 공격", "Weapon Attack"), ("임시변통 무기", "Improvised Weapons"),
    ("도달 거리", "Reach"), ("영웅적 영감", "Heroic Inspiration"), ("내림", "Round Down"),
    ("동시 효과", "Simultaneous Effects"),
    # ── 능력치 ──
    ("근력", "Strength"), ("민첩", "Dexterity"), ("건강", "Constitution"),
    ("지능", "Intelligence"), ("지혜", "Wisdom"), ("카리스마", "Charisma"),
    # ── 상태이상 15 ──
    ("실명", "Blinded"), ("매혹", "Charmed"), ("청각 상실", "Deafened"), ("탈진", "Exhaustion"),
    ("겁에 질림", "Frightened"), ("붙잡힘", "Grappled"), ("행동 불능", "Incapacitated"), ("투명", "Invisible"),
    ("마비", "Paralyzed"), ("석화", "Petrified"), ("중독", "Poisoned"), ("넘어짐", "Prone"),
    ("포박", "Restrained"), ("기절", "Stunned"), ("무의식", "Unconscious"),
    # ── 행동 12 ──
    ("공격 행동", "Attack action"), ("질주 행동", "Dash action"), ("이탈 행동", "Disengage action"),
    ("회피 행동", "Dodge action"), ("도움 행동", "Help action"), ("은신 행동", "Hide action"),
    ("영향 행동", "Influence action"), ("마법 행동", "Magic action"), ("대기 행동", "Ready action"),
    ("수색 행동", "Search action"), ("연구 행동", "Study action"), ("사용 행동", "Utilize action"),
    # ── 이동·환경 ──
    ("험지", "Difficult Terrain"), ("기어가기", "Crawling"), ("오르기", "Climbing"), ("수영", "Swimming"),
    ("높이뛰기", "High Jump"), ("멀리뛰기", "Long Jump"), ("부양", "Hover"), ("낙하", "Falling"),
    ("등반 속도", "Climb Speed"), ("비행 속도", "Fly Speed"), ("수영 속도", "Swim Speed"),
    ("굴착 이동 속도", "Burrow Speed"), ("점유 공간", "Occupied Space"), ("빈 공간", "Unoccupied Space"),
    ("운반 용량", "Carrying Capacity"), ("탈수", "Dehydration"), ("영양실조", "Malnutrition"),
    ("질식", "Suffocation"), ("불붙음", "Burning"), ("물체 부수기", "Breaking Objects"),
    # ── 시야·감각 ──
    ("밝은 빛", "Bright Light"), ("희미한 빛", "Dim Light"), ("어둠", "Darkness"),
    ("약간 가려짐", "Lightly Obscured"), ("심하게 가려짐", "Heavily Obscured"),
    ("암시야", "Darkvision"), ("맹시", "Blindsight"), ("진동감지", "Tremorsense"), ("진시", "Truesight"),
    ("텔레파시", "Telepathy"), ("수동 감지", "Passive Perception"),
    # ── 사회·태도 ──
    ("성향", "Alignment"), ("태도", "Attitude"), ("우호적", "Friendly"), ("무관심", "Indifferent"), ("적대적", "Hostile"),
    # ── 피해 종류 13 ──
    ("산성", "Acid"), ("타격", "Bludgeoning"), ("냉기", "Cold"), ("화염", "Fire"), ("역장", "Force"),
    ("번개", "Lightning"), ("사령", "Necrotic"), ("관통", "Piercing"), ("독", "Poison"), ("정신", "Psychic"),
    ("광휘", "Radiant"), ("참격", "Slashing"), ("천둥", "Thunder"),
    ("피해 유형", "Damage Types"), ("피해 임계값", "Damage Threshold"),
    # ── 주문 ──
    ("캔트립", "Cantrip"), ("의식", "Ritual"), ("주문 시전 매개체", "Spellcasting Focus"),
    ("시전 시간", "Casting Time"), ("사거리", "Range"), ("구성요소", "Components"), ("지속 시간", "Duration"),
    ("음성", "Verbal"), ("동작", "Somatic"), ("물질", "Material"),
    ("효과 범위", "Area of Effect"), ("원뿔", "Cone"), ("정육면체", "Cube"), ("원기둥", "Cylinder"),
    ("발산", "Emanation"), ("직선", "Line"), ("구", "Sphere"), ("마법 효과", "Magical Effect"),
    ("순간이동", "Teleportation"), ("변신", "Shape-Shifting"), ("저주", "Curses"),
    ("방호", "Abjuration"), ("소환", "Conjuration"), ("예지", "Divination"), ("환혹", "Enchantment"),
    ("방출", "Evocation"), ("환영", "Illusion"), ("강령", "Necromancy"), ("변환", "Transmutation"),
    # ── 클래스 12 ──
    ("바바리안", "Barbarian"), ("바드", "Bard"), ("클레릭", "Cleric"), ("드루이드", "Druid"),
    ("파이터", "Fighter"), ("몽크", "Monk"), ("팔라딘", "Paladin"), ("레인저", "Ranger"),
    ("로그", "Rogue"), ("소서러", "Sorcerer"), ("워록", "Warlock"), ("위저드", "Wizard"),
    # ── 클래스 기능 ──
    ("분노", "Rage"), ("암습", "Sneak Attack"), ("야생 변신", "Wild Shape"), ("추가 공격", "Extra Attack"),
    ("바드의 영감", "Bardic Inspiration"), ("신성 변환", "Channel Divinity"), ("안수", "Lay On Hands"),
    ("메타매직", "Metamagic"), ("섬뜩한 기원", "Eldritch Invocations"), ("계약 마법", "Pact Magic"),
    ("행동 쇄도", "Action Surge"), ("재기의 바람", "Second Wind"), ("무기 마스터리", "Weapon Mastery"),
    # ── 무기 마스터리 속성 ──
    ("쪼개기", "Cleave"), ("스치기", "Graze"), ("연속 공격", "Nick"), ("밀치기", "Push"),
    ("약화", "Sap"), ("둔화", "Slow"), ("넘어뜨리기", "Topple"), ("괴롭히기", "Vex"),
    # ── 재주 ──
    ("경계", "Alert"), ("마법 입문", "Magic Initiate"), ("야만적 공격자", "Savage Attacker"),
    ("다재다능", "Skilled"), ("그래플러", "Grappler"), ("능력치 향상", "Ability Score Improvement"),
    ("궁술", "Archery"), ("대형 무기 전투", "Great Weapon Fighting"), ("쌍수 전투", "Two-Weapon Fighting"),
    # ── 기타 규칙 ──
    ("조율", "Attunement"), ("방어구 숙련", "Armor Training"), ("도전 지수", "Challenge Rating"),
    ("경험치", "Experience Points"), ("생물 유형", "Creature Type"), ("스탯 블록", "Stat Block"),
]

# 같은 한국어가 두 번 들어가면 뒤의 것이 앞을 덮어써서 → 미리 확인
seen = set()
for ko, en in TERM_V2:
    assert ko not in seen, f"한국어 중복: {ko}"
    seen.add(ko)

TERM_MAP = {}
for ko, en in TERM_V2:
    TERM_MAP[ko] = en
print("용어표:", len(TERM_MAP))

# 번역②에 보여 줄 용어: 용어표에서 뽑기 (예전 27개와 같은 순서)
CONDITIONS = ["실명", "매혹", "청각 상실", "탈진", "겁에 질림", "붙잡힘", "행동 불능", "투명",
              "마비", "석화", "중독", "넘어짐", "포박", "기절", "무의식"]
ACTIONS = ["공격 행동", "질주 행동", "이탈 행동", "회피 행동", "도움 행동", "은신 행동",
           "영향 행동", "마법 행동", "대기 행동", "수색 행동", "연구 행동", "사용 행동"]

RULE_TERMS = []
for ko in CONDITIONS + ACTIONS:
    RULE_TERMS.append(TERM_MAP[ko])
print("번역② 용어:", len(RULE_TERMS))

def glossary(text):                 # 글에 나오는 용어만 골라 "영어=한국어" 목록 만들기 (GM 프롬프트용)
    out = []
    for ko, en in TERM_MAP.items():
        if " " in en:
            # 두 단어 이상 (예: Attack Roll): 복수형 s 허용, 대소문자 무시
            pattern = r"\b" + re.escape(en) + r"s?\b"
            found = re.search(pattern, text, re.I)
        else:
            # 한 단어 (예: Prone): 대소문자까지 정확히 (일반 단어와 구분)
            pattern = r"\b" + re.escape(en) + r"\b"
            found = re.search(pattern, text)
        if found:
            out.append(f"{en}={ko}")
    if not out:
        return "(없음)"
    return ", ".join(out)


print(glossary("You have Disadvantage while Prone. Take the Dash action."))

# 용어표: 200
# 번역② 용어: 27
# Disadvantage=불리점, Prone=넘어짐, Dash action=질주 행동

## 4. 번역 (한국어 질문 → 영어 검색어)
- 번역①: 일반 번역 (예시 3개)
- 번역②: 규칙 용어 27개(상태이상 15 + 행동 12) 중에 맞는 게 있으면 그대로 쓰기
- 호출은 **14B, temperature 0, seed 42, num_ctx 8192**

In [ ]:
# Ex06. 번역 호출

def ask(prompt, model=None, temperature=0.0):      # 번역용 LLM 호출 (항상 같은 결과)
    options = {
        "temperature": temperature,        # 0 = 가장 확실한 답만
        "seed": GEN_SEED,
        "num_ctx": NUM_CTX,
    }
    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or TRANSLATE_MODEL,       # model을 안 주면 기본 번역 모델
              "messages": [{"role": "user", "content": prompt}],
              "stream": False,
              "think": False,                          # 생각 모드 끄기 (qwen3 이상용, 2.5에는 영향 없음)
              "options": options},
        timeout=300,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]

In [ ]:
# Ex07. 번역 프롬프트

def clean(text):                    # 번역 결과 다듬기 (첫 줄만, "Query:"·따옴표·영어 아닌 글자 제거)
    text = text.strip().split("\n")[0]                       # 첫 줄만
    text = re.sub(r"^\s*query\s*:\s*", "", text, flags=re.I) # 앞의 "Query:" 지우기 (대소문자 무시)
    text = re.sub(r"[\u3040-\u30ff\u3400-\u9fff\uac00-\ud7af]+", " ", text)   # 한자·일본어·한글 → 공백 (예: "Poison中毒效果" → "Poison")
    text = re.sub(r"\s+", " ", text)                          # 공백 여러 개 → 하나
    return text.strip().strip("\"'")                         # 양 끝 따옴표 지우기


# ── 번역① : 일반 번역 (예시 3개를 보여 주고 따라 하게) ──
REWRITE_TEMPLATE = """Convert the user's question about D&D 5.2 rules into a short English search query.
Use the official rule terms that the rulebook would use.
Output only the query, nothing else.

Question: 숙련 보너스는 레벨마다 얼마야?
Query: Proficiency Bonus by character level

Question: 레벨업하려면 경험치가 얼마나 필요해?
Query: Experience Points needed to advance level

Question: 방어도는 어떻게 계산해?
Query: Armor Class calculation

Question: {question}
Query:"""


def translate1(question):           # 번역① 일반 번역
    prompt = REWRITE_TEMPLATE.format(question=question)
    return clean(ask(prompt))


# ── 번역② : 규칙 용어 27개 중에 맞는 게 있으면 그대로 쓰게 ──
# 용어 목록(RULE_TERMS)은 Ex05 용어표에서 만들어요
REWRITE2 = """Translate the user's D&D question into a short English search query for the D&D rules.
If the question is about one of these official rule terms, use that term exactly as written.
Official terms: {terms}
Output only the query, nothing else.

Question: {q}
Query:"""


def translate2(question):           # 번역② 규칙 용어 선택지
    terms = ", ".join(RULE_TERMS)
    prompt = REWRITE2.format(terms=terms, q=question)
    return clean(ask(prompt))


print(translate1("겁에 질리면 뭘 못 해?"), "|", translate2("겁에 질리면 뭘 못 해?"))

# Effects of being Frightened | frightened action restrictions

## 5. 키워드 검색 BM25
임베딩은 **뜻**, BM25는 **글자 그대로의 단어**를 비교해서 서로 보완해요.

In [ ]:
# Ex08. 키워드 검색

# 불용어: 너무 흔해서 검색에 도움이 안 되는 단어
STOP = set("a an the of to in on and or is are be can you your it its as at by for with that this if when how what do does".split())


def tokenize(text):                 # 글 → 단어 목록 (소문자, 영어·숫자만, 불용어 제외)
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [w for w in words if w not in STOP]


class BM25:
    # 점수가 높아지는 경우: 단어가 많이 나옴(tf) + 드문 단어(idf) + 짧은 청크(len)

    def __init__(self, docs, k1=1.5, b=0.75):      # 청크들의 단어 통계 미리 계산
        self.k1 = k1                               # 빈도가 늘 때 점수가 오르는 정도
        self.b = b                                 # 길이 보정을 얼마나 할지
        self.tf = [Counter(d) for d in docs]       # 청크마다 단어별 등장 횟수
        self.len = np.array([len(d) for d in docs], dtype="float32")   # 청크마다 단어 수
        self.avg = self.len.mean()                 # 평균 단어 수

        # df = 그 단어가 나오는 청크 수 (한 청크에서 여러 번 나와도 1번으로)
        df = Counter()
        for d in docs:
            for w in set(d):
                df[w] += 1

        # idf = 드문 단어일수록 큰 값
        total = len(docs)                          # 전체 청크 수
        self.idf = {}
        for w, n in df.items():
            self.idf[w] = math.log(1 + (total - n + 0.5) / (n + 0.5))

    def scores(self, query):                       # 질문 → 청크마다 점수
        s = np.zeros(len(self.tf), dtype="float32")
        for w in set(tokenize(query)):
            if w not in self.idf:                  # 룰북에 없는 단어는 건너뜀
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(w, 0)                   # 이 청크에서 단어가 나온 횟수
                if f:
                    scale = 1 - self.b + self.b * self.len[i] / self.avg   # 길이 보정 (긴 청크일수록 큼)
                    s[i] += self.idf[w] * f * (self.k1 + 1) / (f + self.k1 * scale)
                    # 희귀도 × 빈도 효과 / 길이 보정
        return s


# 제목을 두 번 넣어서 제목 단어에 가중치
docs = []
for c in chunks:
    title = c["meta"]["title"]
    docs.append(tokenize(title + " " + title + " " + c["text"]))
bm25 = BM25(docs)


def keyword(question, k=10):        # 키워드 검색 → 점수 높은 청크 k개
    order = np.argsort(-bm25.scores(question))            # 점수 큰 순서의 번호 (-를 붙여 내림차순)
    return [chunks[i] for i in order[:k]]


print([c["meta"]["title"] for c in keyword("Two-Weapon Fighting", 3)])

# ['Two-Weapon Fighting', 'Great Weapon Fighting', 'Level 2: Fighting Style']

## 6. 최종 검색 `retrieve` (임베딩 3갈래 + 키워드 2갈래 → 가중 RRF)

In [ ]:
# Ex09. 최종 검색

def semantic(texts, k=5):           # 임베딩 검색 (뜻으로 찾기: 글 여러 개 → 비슷한 청크 k개)
    vectors = embed(texts)
    res = collection.query(query_embeddings=vectors, n_results=k)

    # 같은 청크가 여러 번 나오면 가장 높은 점수만 남김
    best = {}
    for ids, dists in zip(res["ids"], res["distances"]):     # 글마다 결과 k개
        for key, dist in zip(ids, dists):
            i = int(key)                        # Chroma id = 청크 순서 번호 (문자열)
            score = 1 - dist                    # 코사인 거리 → 유사도 (클수록 비슷)
            best[i] = max(best.get(i, -1.0), score)

    order = sorted(best, key=best.get, reverse=True)   # 점수 높은 순
    return [chunks[i] for i in order[:k]]

def semantic2(texts, k=5):      # 임베딩 검색을 글마다 따로 (임베딩 호출은 한 번에)
    vectors = embed(texts)       # 글 여러 개를 한 번의 호출로 벡터로
    res = collection.query(query_embeddings=vectors, n_results=k)

    lists = []
    for ids in res["ids"]:                  # 글마다 결과 k개 (가까운 순)
        found = []
        for key in ids:
            found.append(chunks[int(key)])  # Chroma id = 청크 순서 번호
        lists.append(found)
    return lists

def merge(lists, weights, k=TOP_K, c0=RRF_C):     # 여러 검색 결과를 순위 점수(RRF)로 합치기
    # 점수 = 비중 / (c0 + 순위) → 여러 목록에서 위에 있을수록 점수가 높아짐
    score, lookup = {}, {}
    for items, w in zip(lists, weights):
        for r, c in enumerate(items, 1):           # r = 순위 (1부터)
            score[c["id"]] = score.get(c["id"], 0) + w / (c0 + r)
            lookup[c["id"]] = c                    # id → 청크 (마지막에 되찾으려고)

    order = sorted(score, key=score.get, reverse=True)
    return [lookup[i] for i in order[:k]]

pool = ThreadPoolExecutor(max_workers=NUM_PARALLEL) # 동시요청

def retrieve(question, k=TOP_K):                        # 최종 검색
    job1 = pool.submit(translate1, question)            # 번역1
    job2 = pool.submit(translate2, question)            # 번역2
    job0 = pool.submit(semantic, [question], POOL_K)    # 임베딩 : 원문 질문

    # 끝날 때까지 기다렸다가 결과 받기
    en1 = job1.result()
    en2 = job2.result()
    first = job0.result()

    pair = semantic2([en1, en2], POOL_K)             # 임베딩: 번역1, 2를 한 번에

    lists = [
        first,                                       # 임베딩: 원문 질문
        pair[0],                                     # 임베딩: 번역1
        pair[1],                                     # 임베딩: 번역2
        keyword(en1, POOL_K),                        # 키워드: 번역1
        keyword(en2, POOL_K),                        # 키워드: 번역2
    ]
    return merge(lists, SEARCH_WEIGHTS, k), [question, en1, en2]    # (청크 5개, 검색에 쓴 글 3개)

In [ ]:
# 메모. 검색 개선 후보
# 비중 조정 → [1,1,1,0.5,0.5]를 다른 조합으로 변경 (BM25 0.3/0.7, 번역2 가중 상향 등)
# RRF 상수 → c0=60을 20/100으로 변경
# 후보 폭 → 목록당 10개를 20개로 확대
# 번역2 프롬프트 → "확실하지 않으면 용어 넣지 말기" + 핵심 명사 유지 + 예시 1~2개
# 번역 횟수 → 번역③(키워드만 추출) 추가
# BM25 제목 가중 → 제목 2번을 1번·3번으로 변경
# 불용어 → 목록 확대 또는 제거
# 청크 분할 → 긴 청크를 나누거나 짧은 청크를 합침
# 임베딩 입력 → 청크에 제목·경로를 붙여서 임베딩
# 리랭커 → 재도입 (이미 하락, 기각 상태)

## 7. 번역·검색 캐시 (개발용, 꺼 둠)
청크나 검색 방식을 바꾸는 실험 중에는 옛 결과가 나올 수 있어서 꺼 뒀어요.

In [ ]:
# Ex10-a. 캐시 장치 (꺼 둠)
# 같은 질문을 다시 물으면 모델을 부르지 않고 저장해 둔 결과를 바로 돌려줌. -> 개발/실험용

# by_id = {c["id"]: c for c in chunks}     # 청크 id → 청크 (파일에서 불러올 때 id로 청크를 되찾으려고)

# _tr_cache = {}   # 번역 기억장:  ("1", 질문) 또는 ("2", 질문)  →  영어 검색어
# _sf_cache = {}   # 검색 기억장:  (질문, 개수)                 →  (청크 목록, [질문, 번역①, 번역②])


# # ── 1. 원래 함수 보관 ─────────────────────────────────────────
# # 셀을 두 번 돌려도 "캐시 함수를 또 캐시로 감싸는" 일이 없게, 처음 한 번만 보관.
# if "_orig_tr1" not in globals():
#     _orig_tr1 = translate1
#     _orig_tr2 = translate2
#     _orig_retrieve = retrieve


# # ── 2. 같은 이름으로 캐시 버전 만들기 ─────────────────────────
# # 이름이 같아서, 다른 셀(Ex07 retrieve 안의 번역 호출 등)도 자동으로 캐시 버전을 써요.
# def translate1(q):                          # 번역① (캐시 버전)
#     key = ("1", q)
#     if key not in _tr_cache:              # 처음 보는 질문일 때만 모델 호출
#         _tr_cache[key] = _orig_tr1(q)
#     return _tr_cache[key]


# def translate2(q):                          # 번역② (캐시 버전)
#     key = ("2", q)
#     if key not in _tr_cache:
#         _tr_cache[key] = _orig_tr2(q)
#     return _tr_cache[key]


# def retrieve(q, k=TOP_K):                  # 검색 (캐시 버전)
#     key = (q, k)
#     if key not in _sf_cache:
#         _sf_cache[key] = _orig_retrieve(q, k)
#     return _sf_cache[key]


# # ── 3. 드라이브에 저장 / 불러오기 ─────────────────────────────
# # JSON은 튜플 키를 못 쓰니까 "1|질문", "질문|5"처럼 |로 이어 붙인 글자로 저장해요.
# # 청크는 통째로 저장하지 않고 id만 저장해요 (불러올 때 by_id로 되찾음).
# def save():                                 # 캐시를 드라이브에 저장
#     tr = {}
#     for (kind, q), en in _tr_cache.items():
#         tr[f"{kind}|{q}"] = en

#     sf = {}
#     for (q, k), (found, texts) in _sf_cache.items():
#         sf[f"{q}|{k}"] = {"ids": [c["id"] for c in found], "texts": texts}

#     with open(CACHE_PATH, "w", encoding="utf-8") as f:
#         json.dump({"tr": tr, "sf": sf}, f, ensure_ascii=False)
#     print("저장:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")


# def restore():                              # 드라이브에서 캐시 불러오기
#     if not os.path.exists(CACHE_PATH):
#         print("캐시 파일 없음 → 아래 build(dev)로 만들어요")
#         return

#     with open(CACHE_PATH, encoding="utf-8") as f:
#         d = json.load(f)

#     for key, en in d["tr"].items():
#         kind, q = key.split("|", 1)            # 앞에서 한 번만 자름: "1|질문"
#         _tr_cache[(kind, q)] = en

#     for key, v in d["sf"].items():
#         q, k = key.rsplit("|", 1)              # 뒤에서 한 번만 자름: "질문|5" (질문에 |가 있어도 안전)
#         found = [by_id[i] for i in v["ids"]]
#         _sf_cache[(q, int(k))] = (found, v["texts"])

#     print("불러옴:", len(_tr_cache), "번역 /", len(_sf_cache), "검색")


# # ── 4. 모델 내리기 ───────────────────────────────────────────
# # keep_alive 0 = 지금 바로 GPU 메모리에서 내림 (14B ↔ bge-m3 교체를 줄이려고)
# def unload(model):                          # 모델을 GPU 메모리에서 내리기
#     requests.post(f"{OLLAMA_URL}/api/generate", json={"model": model, "keep_alive": 0})


# # ── 실행 ────────────────────────────────────────────────────
# dev = devset
# restore()

# 불러옴: 132 번역 / 66 검색

In [ ]:
# Ex10-b. 캐시 만들기 (꺼 둠)

# def build(items):             # 질문 세트의 번역·검색 결과를 미리 만들어 두기
#     qs = [g["q"] for g in items]

#     # 1. 번역: 번역①② 중 하나라도 없는 질문만
#     todo_tr = []
#     for q in qs:
#         if ("1", q) not in _tr_cache or ("2", q) not in _tr_cache:
#             todo_tr.append(q)

#     if todo_tr:
#         _orig_tr1(todo_tr[0])                      # 워밍업: 14B를 메모리에 올리기 (결과는 버림)
#         for q in tqdm(todo_tr, desc="번역(14B)"):
#             translate1(q)
#             translate2(q)
#         unload(TRANSLATE_MODEL)                    # 14B 내리기 → 임베딩 모델 자리 비우기

#     # 2. 검색: 검색 결과가 없는 질문만
#     todo_sf = []
#     for q in qs:
#         if (q, TOP_K) not in _sf_cache:
#             todo_sf.append(q)

#     for q in tqdm(todo_sf, desc="검색(bge-m3)"):
#         retrieve(q)
#     if todo_sf:
#         unload(EMBED_MODEL)

#     print(f"완료. 번역 새로 {len(todo_tr)}개, 검색 새로 {len(todo_sf)}개 → 캐시 번역 {len(_tr_cache)} / 검색 {len(_sf_cache)}")


# build(dev)
# save()

# 검색(bge-m3): 0/0 [00:00<?, ?it/s]
# 완료. 번역 새로 0개, 검색 새로 0개 → 캐시 번역 132 / 검색 66
# 저장: 132 번역 / 66 검색

## 8. LLM 호출 (근거 선택·검증용 짧은 답)

In [ ]:
# Ex11. LLM 호출

# 시스템 프롬프트
SYSTEM = ("You are a D&D rules assistant.")

def chat(prompt, model=None, system=SYSTEM, temperature=0.0, seed=GEN_SEED, limit=None):   # LLM 호출 (기본)
    options = {"num_ctx": NUM_CTX, "temperature": temperature}
    if seed is not None:
        options["seed"] = seed                     # seed가 있으면 같은 결과로 고정
    if limit:
        options["num_predict"] = limit             # 출력 길이 제한 (짧은 답용)

    res = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={"model": model or GEN_MODEL,
              "messages": [{"role": "system", "content": system},
                           {"role": "user", "content": prompt}],
              "stream": False,
              "think": False,                      # 생각 모드 끄기 (qwen3 이상용)
              "options": options},
        timeout=600,
    )
    res.raise_for_status()
    return res.json()["message"]["content"]


def ask3(prompt, model=None, system=SYSTEM):         # 짧은 답용 호출 (번호, 예/아니오) — 최대 20토큰
    return chat(prompt, model, system, 0.0, GEN_SEED, limit=20)

## 9. 근거 파이프라인 (LangGraph)

```
START → search(번역①② 동시 + 검색) → choose(근거 번호) → check(검증) ┬ 통과 → accept(원본 전체 근거) → END
                                                                    └ 실패 → refuse(거절) → END
```
- check: 고른 근거를 먼저 검증 → 실패하면 나머지 4개를 동시에 검증 → 검색 순위가 높은 통과 후보
- 한국어 설명은 GM이 써요

In [ ]:
# Ex12. 근거 선택
SELECT_TEMPLATE = """다음은 D&D 규칙 질문과 후보 근거입니다. 질문에 직접 답하는 근거의 번호 하나만 고르세요.
- 질문이 묻는 규칙 자체를 설명하는 근거를 고르세요. 이름만 비슷하거나 예시로 지나가듯 언급되는 근거는 고르지 마세요.
- 숫자 하나만 출력하세요.

### 질문
{question}

### 후보 근거
{cands}

### 번호"""


def select(question, found, model=None):   # 후보 중 질문에 맞는 근거 번호 고르기 (1부터)
    # 후보마다 "[번호] 제목: 본문 앞부분 300자" 한 줄씩
    lines = []
    for n, c in enumerate(found, 1):
        preview = c["text"][:SELECT_CHARS].replace("\n", " ")   # 줄바꿈 → 공백 (한 줄로)
        lines.append(f"[{n}] {c['meta']['title']}: {preview}")
    cands = "\n".join(lines)

    out = ask3(SELECT_TEMPLATE.format(question=question, cands=cands), model)

    # 답에서 첫 숫자 꺼내기. 숫자가 없거나 범위 밖이면 1번
    m = re.search(r"\d+", out)
    n = int(m.group()) if m else 1
    if 1 <= n <= len(found):
        return n
    return 1

In [ ]:
# Ex13. 원본 전체 읽기
# 검색은 나뉜 조각으로, GM에게 넘길 근거는 원본 전체로

def heading(chunk):                 # 청크 머리줄 (장 > 절 > 제목)
    return chunk["text"].split("\n", 1)[0]


def content(chunk):                 # 청크 본문 (둘째 줄부터)
    return chunk["text"].split("\n", 1)[1]


# 나뉜 조각끼리 모아 두기: (제목, 쪽) → {조각 번호: 청크}
parts = {}
for c in chunks:
    meta = c["meta"]
    if "part" not in meta:                      # 나뉘지 않은 청크는 건너뜀
        continue
    key = (meta["title"], meta["page"])
    if key not in parts:
        parts[key] = {}
    parts[key][meta["part"]] = c


def expand(chunk):                  # 나뉜 조각이면 원본 전체(모든 조각)를 이어서 돌려줌
    meta = chunk["meta"]
    if "part" not in meta:                      # 나뉘지 않은 청크는 그대로
        return chunk

    group = parts[(meta["title"], meta["page"])]

    texts = []
    for k in range(1, meta["parts"] + 1):       # 1번 조각부터 마지막 조각까지 순서대로
        texts.append(content(group[k]))
    whole = " ".join(texts)

    return {"id": chunk["id"], "text": heading(chunk) + "\n" + whole, "meta": meta}

In [ ]:
# Ex14. 근거 검증
VERIFY_TEMPLATE = """다음 D&D 규칙 질문과 근거입니다. 이 근거가 질문에 직접 답하는 규칙 내용을 담고 있으면 '예', 이름만 비슷하거나 다른 주제이면 '아니오'라고만 답하세요.
질문은 한국어, 근거는 영어예요. 표현이 달라도 같은 규칙을 다루면 '예'예요.

### 질문
{question}
(영어 검색어: {en})

### 근거
{title}: {text}

### 답 (예/아니오)"""

def verify(question, chunk, english=""):     # 근거가 질문에 맞는지 LLM에게 묻기 (예 → True)
    prompt = VERIFY_TEMPLATE.format(question=question, en=english or "-", title=chunk["meta"]["title"],
                                    text=chunk["text"][:VERIFY_CHARS])
    out = ask3(prompt)
    out = re.sub(r"[\s\*'\"]", "", out)               # 공백·별표·따옴표 지우기 ("**예**" → "예")
    return out.startswith(("예", "네"))

def match(texts, chunk):            # 근거 제목이 번역①② 안에 그대로 있는지 (있으면 확실)
    t = norm(chunk["meta"]["title"])
    for x in texts[1:]:                                # texts = [원문, 번역①, 번역②]
        if re.search(r"\b" + re.escape(t) + r"\b", x.lower()):
            return True
    return False

def verify2(question, chunk, texts):    # 근거 통과 여부 (제목이 맞으면 바로 통과, 아니면 LLM 검증)
    return match(texts, chunk) or verify(question, chunk, texts[1])

def pick(question, found, texts, n):    # 고른 근거부터 검증해서 처음 통과한 번호 (모두 실패면 None)
    # 1단계 : 고른 근거만 먼저 검증 (대부분 여기서 끝남)
    if verify2(question, found[n - 1], texts):
        return n

    # 2단계 : 나머지 후보를 검색 순위대로 모아서 동시에 검증
    others = []
    for i in range(1, len(found) + 1):
        if i != n:                                  # 1단계에서 실패한 번호는 빼고
            others.append(i)                        # 나머지는 검색 순위대로

    jobs = []
    for i in others:
        jobs.append(pool.submit(verify2, question, found[i - 1], texts))

    # 순위가 높은 후보부터 결과 확인 -> 처음 통과한 번호 (예전과 같은 규칙)
    for i, job in zip(others, jobs):
        if job.result():
            return i
    return None

In [ ]:
# Ex15. 최종 함수 (LangGraph)
# 질문 → 검색 → 근거 선택 → 검증 → 근거 or 거절

class State(TypedDict, total=False):    # 노드끼리 주고받는 값
    question: str                       # 질문
    found: list                         # 검색 후보 5개
    texts: list                         # [원문, 번역1, 번역2]
    n: int                              # 고른 번호 → 검증 후 통과 (모두 실패면 None)
    result: dict                        # 최종 결과

def search(state):          # 노드: 검색
    found, texts = retrieve(state["question"])
    return {"found": found, "texts": texts}

def choose(state):          # 노드: 후보 중 근거 번호 고르기
    n = select(state["question"], state["found"])
    return {"n": n}

def check(state):           # 노드: 고른 근거부터 검증 (실패하면 나머지 동시 검증)
    n = pick(state["question"], state["found"], state["texts"], state["n"])
    return {"n": n}

def route(state):           # 갈림길: 통과 → accept, 모두 실패 → refuse
    if state["n"] is None:
        return "refuse"
    return "accept"

def accept(state):          # 노드: 검증된 근거를 GM에게 넘길 모양으로
    c = expand(state["found"][state["n"] - 1])     # 나뉜 청크면 원본 전체
    result = {
        "status": "ok",
        "title": c["meta"]["title"],
        "page": c["meta"]["page"],
        "path": heading(c),                        # 장 > 절 > 제목
        "text": content(c),                        # 영어 본문 (GM 프롬프트에 넣을 근거)
        "terms": glossary(c["text"]),              # 근거에 나온 용어 "영어=한국어"
        "picked": state["n"],                      # 후보 중 몇 번째였는지
        "queries": state["texts"][1:],             # 번역1, 2
    }
    return {"result": result}

def refuse(state):
    candidates = []
    for x in state["found"][:3]:
        candidates.append((x["meta"]["title"], x["meta"]["page"]))
    result = {
        "status": "refuse",
        "candidates": candidates,
        "queries": state["texts"][1:],
    }
    return {"result": result}

# 그래프 만들기 노드등록 -> 연결 -> 컴파일
graph = StateGraph(State)
graph.add_node("search", search)
graph.add_node("choose", choose)
graph.add_node("check", check)
graph.add_node("accept", accept)
graph.add_node("refuse", refuse)

graph.add_edge(START, "search")
graph.add_edge("search", "choose")
graph.add_edge("choose", "check")
graph.add_conditional_edges("check", route, ["accept", "refuse"])   # 검증 결과에 따라 갈림
graph.add_edge("accept", END)
graph.add_edge("refuse", END)

app = graph.compile()

def evidence(question):         # 질문에 답하는 룰북 근거 찾기 (GM이 부르는 함수)
    state = app.invoke({"question": question})
    return state["result"]

### 메모. LangGraph 확장 후보

**현재 그래프**: START → search → choose → check ┬ 통과 → accept → END
                                                  └ 실패 → refuse → END

| 순서 | 기법 | 하는 일 | 시간 영향 | 품질 영향 | 주의 |
|---|---|---|---|---|---|
| 1 | 그래프 그림 (`app.get_graph().draw_mermaid_png()`) | 흐름도 이미지 생성 (팀 공유·문서용) | 없음 | 없음 | - |
| 2 | 노드 캐시 (`CachePolicy`) | `search` 노드 결과 저장 → 같은 질문은 번역·검색 건너뜀 (Ex10 캐시 대체) | 반복 질문 약 0.6초 → 거의 0 | 그대로 | 청크·번역 방식을 바꾸면 반드시 캐시 비우기 (옛 결과 섞임) |
| 3 | 재시도 루프 (되돌아가는 간선 + 횟수 제한 1번) | 거절 시 다른 방식 번역(핵심 명사만)으로 한 번 더 검색 | 거절 질문만 약 +1초 | 검색 실패 6개 중 일부 회복 가능 | 모든 질문에 같은 규칙 (특정 용어 넣기 금지), 실험2로 효과·시간 같이 측정 |
| 4 | 되묻기 (`interrupt`) | 거절 시 멈추고 후보 3개 중 고르게 함 | 사람 답 대기 | 애매한 질문을 사람이 해결 | 화면(채팅 UI)이 있어야 의미 있음 |
| 5 | 진행 상황 표시 (`stream`) | 노드가 끝날 때마다 단계 표시 | 실제 시간 그대로 (체감만 줄어듦) | 그대로 | 화면이 있어야 의미 있음 |
| 6 | 분기 실행 (`Send`) | 나머지 후보 검증을 노드 여러 개로 갈라 그래프에 표시 | 그대로 (`pick`이 이미 동시 실행) | 그대로 | 동시 실행을 두 군데서 관리하게 됨 → 보류 |

In [ ]:
# Ex16. 화면 출력

def show(question):                 # 근거를 화면에 출력 (개발 확인용)
    e = evidence(question)

    if e["status"] == "refuse":
        print("🙅 맞는 규칙 근거를 찾지 못했어요. 후보:", e["candidates"])
        print("   번역:", e["queries"])
        return e

    print(f"출처: {e['title']} p.{e['page']} ({e['path']})")
    print("용어:", e["terms"])
    print("근거:", e["text"][:600])
    return e

### 질문해 보기 (여기만 바꿔서 다시 실행)

In [ ]:
question = "넘어졌을 때 일어나려면 이동력이 얼마나 들어?"
_ = show(question)

## 📝 실험 기록

**검색**
- 번역을 7B → 14B로 바꿔 개발셋 검색을 다시 쟀다 → 66문항 합계 (43, 52, 59)로 7B (41, 54, 57)와 비슷.
- 번역 방식 A~E 비교 → 세 갈래 RRF(E)가 가장 좋아서 채택.
- 키워드(BM25) 합치기 비교 → 키워드 비중 0.5(F2)를 검색 v1로 채택.
- 번역①의 용어 힌트를 빼 봤다 → 점수가 비슷해서 힌트 없는 일반 번역으로 확정.
- 한국어 + 번역 두 번 검색해서 합치기 → 이득 없어서 기각.
- 재순위(reranker) 실험 → 1등이 오히려 줄어서(31 → 28) 기각. -> 재시도
- Prone·Hide 질문은 어떤 방식으로도 잘 못 찾음 → 알려진 약점. -> 더 있을지도?

**답변**
- 근거 선택을 "한 번에 하기" vs "2단계(번호 먼저)"로 비교 → 2단계 채택.
- 근거 검증(예/아니오) 평가 + 거절 정책 비교 → 정책 A(아니오면 바로 거절) 채택.
- 설명 충실성 검사(근거에 없는 내용 묻기)를 추가했다.
- 7B·14B 설명에서 한자 섞임, 분수 뒤집힘(3분의 4) 발견 → 깨짐·혼합·분수 검사와 재생성으로 이어짐.
- 용어표 주입 켬/끔 비교(38문항) → 켬: 정상 31·숨김 3·용어누락 9 / 끔: 정상 28·숨김 6·용어누락 18 → 켬 유지.
- 모델 교체(14B ↔ bge-m3)가 느림의 원인 → 번역·검색 캐시를 만들었다.
- temperature 0 + seed 42로 같은 결과가 나오는지 확인 → 재현됨.

**최종 세트 (golden_final, 60문항)**
- v1 점수는 31 / 46 / 51. 한 번만 재는 시험지라 결과 파일이 있으면 다시 못 재게 잠가 뒀었다.

**2026-11-12 변경 (개발 66문항 기준)**
- FAISS → Chroma(코사인) + 하이픈 살리기 → 검색 (43, 52, 59) → (45, 54, 58)
- 긴 청크 1500자 나누기: 검색 1등 ±1 (잡음 범위), 2000자 넘던 87개 청크를 답변 단계에서 읽을 수 있게 됨
- 나눌 때 겹치기 2문장 → 1등 46 → 44로 오히려 하락 → 기각. 대신 답변 때 원본 전체를 읽게 함(`expand`, NUM_CTX 8192)
- 표 → 문장 변환: PDF에서 띄어쓰기·열이 깨져서 기각
- 모델 비교 (정상/숨김/거절): qwen2.5 14B 55/6/5, qwen3.5 9B 49/8/9, qwen2.5 7B 31/?/17 → 14B 유지
- 캐시는 실험 중 꺼 둠 (청크·검색이 바뀌면 옛 결과가 나오는 문제)

**2026-11-13 변경 (개발 149문항 기준, 정답 있는 136문항)**
- 개발셋 66 → 149문항 (거절 문항 13개 포함)
- 검색 기준: (87, 113, 119)
- `pick`(고른 근거가 아니면 다음 후보도 검증): 정상 101 → 115, 거절 25 → 10, 지어냄 3 그대로 → 채택
- `squash`(띄어쓰기·하이픈 무시 원문 검증): 효과 없음 → 원인이 띄어쓰기가 아니었음
- 용어표 v2: 일반 단어 38개 제거, 상태이상·행동·클래스 기능·마스터리·재주 추가 (200개), 번역② 27개는 용어표에서 뽑음
- 설명 파트(설명 쓰기·검사) 제거 → 룰 RAG는 검증된 근거만 넘기고 설명은 GM이 씀 (질문당 시간: 설명 쓰기 약 3.6초 빠짐)

**2026-11-13 변경 2 (개발 149문항: 정답 있는 137 / 거절 12)**
- 질문 세트 보정: 상위 절 근거 5개를 정답에 추가, 바드 문항 → 정답 있는 문항, 숲 문항 표현 수정, Grappl → Grappled
- 채점: hit2(단수/복수 무시)로 통일, ALIAS(특정 값 예외표) 제거
- 데이터: 깃허브 TRPG-masterbot/data/Min에서 받음 (임베딩 파일만 드라이브)
- 기준값: 검색 (91, 117, 122) / 근거 정답 114 · 다른 근거 13 · 거절 10 / 거절 문항 거절(맞음) 10 · 통과(틀림) 2
- 동시 실행(NUM_PARALLEL 4): 번역①②·원문 임베딩 동시, 검증 2단계(실패 시 나머지 동시) → 결과 그대로, 질문당 약 2초 → 1.7초
- LangGraph로 전환(search → choose → check → accept/refuse) → 결과 그대로
- v1.5 수정: Ex09 임베딩 중복 호출 제거, pick 2단계에서 실패한 번호 재검증 제거, 번역에 섞인 한자 제거(clean), Ex02 return 누락 수정

In [ ]:
# # 실험1. 검색 점수
# # 정답 있는 문항만 채점 (1등 / 3등 안 / 5등 안)
# # 기준: 137문항 (91, 117, 122)
# # 실행하려면: 셀 전체 선택(Ctrl + A) → Ctrl + / 로 주석 해제
# answerable = []
# for g in devset:
#     if g["want"] is not None:                      # 정답 없는 거절 문항은 검색 채점에서 뺌
#         answerable.append(g)

# print("채점 문항:", len(answerable))
# print(evaluate(lambda q: retrieve(q)[0], answerable, misses=False))

In [ ]:
# 실험2. 근거 정확도 · 거절
# 정답 근거 = 검증을 통과한 근거의 제목이 정답과 같음
# 다른 근거 = 통과는 했지만 정답이 아닌 근거 (GM이 틀린 근거로 답하게 됨)
# 거절 문항은 "거절(맞음)"이 많을수록 좋음
# 실행하려면: 셀 전체 선택(Ctrl + A) → Ctrl + / 로 주석 해제
# ask3("안녕")                                       # 워밍업: 14B를 먼저 올려 둠 (시간 측정에서 제외)

# normal = Counter()        # 정답 있는 문항: 정답 근거 / 다른 근거 / 거절
# refusal = Counter()       # 정답 없는 문항: 거절(맞음) / 통과(틀림)
# bytype = {}               # 종류별 결과
# problems = []             # 직접 볼 문항
# t0 = time.time()

# for g in tqdm(devset):
#     e = evidence(g["q"])
#     kind = g.get("type", "기존")
#     if kind not in bytype:
#         bytype[kind] = Counter()

#     if g["want"] is None:                          # 거절해야 맞는 문항
#         if e["status"] == "refuse":
#             result = "거절(맞음)"
#         else:
#             result = "통과(틀림)"
#             problems.append((result, g["q"], e["title"]))
#         refusal[result] += 1
#     else:
#         if e["status"] == "refuse":
#             result = "거절"
#             problems.append((result, g["q"], g["want"], e["candidates"]))
#         elif any(hit2(e["title"], w) for w in wants(g)):
#             result = "정답 근거"
#         else:
#             result = "다른 근거"
#             problems.append((result, g["q"], g["want"], e["title"]))
#         normal[result] += 1
#     bytype[kind][result] += 1

# print(f"[정답 있는 문항] 정답 근거 {normal['정답 근거']} / 다른 근거 {normal['다른 근거']} / 거절 {normal['거절']}")
# print(f"[거절 문항]     거절(맞음) {refusal['거절(맞음)']} / 통과(틀림) {refusal['통과(틀림)']}")
# print("질문당", round((time.time() - t0) / len(devset), 1), "초")

# print("\n[종류별]")
# for kind, c in bytype.items():
#     print(kind, dict(c))

# print("\n[직접 볼 문항]")
# for p in problems:
#     print(p)

In [ ]:
# # 실험3. 단계별 시간 (질문 하나)
# # 설명 쓰기를 뺀 뒤라 검색 + 선택 + 검증만 잼
# # 실행하려면: 셀 전체 선택(Ctrl + A) → Ctrl + / 로 주석 해제
# question = "넘어졌을 때 일어나려면 이동력이 얼마나 들어?"
# ask3("안녕")                                       # 워밍업

# t = time.time()
# found, texts = retrieve(question)
# print("검색 (번역 2 + 검색):", round(time.time() - t, 1), "초")

# t = time.time()
# n = select(question, found)
# print("근거 선택:", round(time.time() - t, 1), "초")

# t = time.time()
# n = pick(question, found, texts, n)
# print("검증:", round(time.time() - t, 1), "초")